# ABSA Baseline Experiments: ATE + ASC (All-in-One)
## Train 6 Models x 2 Tasks trên Kaggle GPU

### Chỉ cần 1 Kaggle Dataset:
- Tạo dataset tên `absa-data` chứa: `train.jsonl`, `dev.jsonl`, `test.jsonl`
- Bật GPU: Settings → Accelerator → GPU T4 x2
- Run All!

Notebook này tự chứa toàn bộ code, không cần upload thêm src/.


## 0. Install & Setup


In [ ]:
import subprocess, sys, os
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'underthesea', 'pytorch-crf', 'gensim'])

import os, json, time, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF
from collections import Counter
from gensim.models import Word2Vec
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from IPython.display import display
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

try:
    from underthesea import word_tokenize
    HAS_UNDERTHESEA = True
except:
    HAS_UNDERTHESEA = False

IS_KAGGLE = os.path.exists('/kaggle/input')
if IS_KAGGLE:
    DATA_DIR = '/kaggle/input/absa-data'
    SAVE_DIR = '/kaggle/working/results'
else:
    DATA_DIR = '../../data'
    SAVE_DIR = '../../results'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42); np.random.seed(42); random.seed(42)
print(f"Device: {device}")
if device.type == 'cuda': print(f"GPU: {torch.cuda.get_device_name(0)}")


## 1. Preprocessing Functions (Embedded)


In [ ]:
# ==================== PREPROCESSING ====================
def load_raw_data(filepath):
    items = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            items.append(json.loads(line.strip()))
    return items

def segment_text(text):
    if HAS_UNDERTHESEA:
        try: return word_tokenize(text, format="text").lower()
        except: return text.lower()
    return text.lower()

def build_vocab(texts, min_freq=2):
    word_freq = Counter(w for t in texts for w in t.split())
    word2idx = {'<PAD>': 0, '<UNK>': 1, '[ASP]': 2}
    for w, freq in word_freq.items():
        if freq >= min_freq:
            word2idx[w] = len(word2idx)
    return word2idx

def tokenize(text, word2idx, max_len=128):
    words = text.split()[:max_len]
    seq = [word2idx.get(w, 1) for w in words]
    length = max(len(seq), 1)
    seq += [0] * (max_len - len(seq))
    return seq, length

def train_w2v(texts, word2idx, emb_dim=150):
    all_sentences = [t.split() for t in texts]
    print(f"Training Word2Vec {emb_dim}d...")
    w2v = Word2Vec(all_sentences, vector_size=emb_dim, window=5,
                   min_count=2, workers=4, epochs=20, sg=1, seed=42)
    vocab_size = len(word2idx)
    emb_matrix = np.random.normal(0, 0.1, (vocab_size, emb_dim)).astype(np.float32)
    emb_matrix[0] = 0
    hit = 0
    for w, idx in word2idx.items():
        if w in w2v.wv: emb_matrix[idx] = w2v.wv[w]; hit += 1
    print(f"Embedding ready. Hit: {hit}/{vocab_size} ({hit/vocab_size*100:.1f}%)")
    return emb_matrix

print("Preprocessing functions loaded!")


## 2. ATE: Tag System & Dataset


In [ ]:
# ==================== ATE TAG SYSTEM ====================
ASPECTS = ["CAMERA","FEATURES","PERFORMANCE","DESIGN","PRICE",
           "GENERAL","SCREEN","BATTERY","STORAGE","SER&ACC"]
O_TAG = 0
BIO_TAGS = ['O']
for a in ASPECTS:
    BIO_TAGS.append(f'B-{a}'); BIO_TAGS.append(f'I-{a}')
TAG2ID = {t:i for i,t in enumerate(BIO_TAGS)}
NUM_ATE_TAGS = len(BIO_TAGS)
print(f"ATE Tags: {NUM_ATE_TAGS}")

def text_to_ate_tags(text, spans, max_len):
    words = text.split()[:max_len]
    positions = []
    pos = 0
    for w in words:
        idx = text.lower().find(w.replace('_',' '), pos)
        if idx == -1: idx = pos
        positions.append((idx, idx + len(w.replace('_',' '))))
        pos = idx + len(w.replace('_',' '))
    tags = [O_TAG] * max_len
    for s, e, raw_label in sorted(spans, key=lambda x: x[1]-x[0]):
        if '#' not in raw_label: continue
        aspect = raw_label.split('#')[0]
        if f'B-{aspect}' not in TAG2ID: continue
        b_id, i_id = TAG2ID[f'B-{aspect}'], TAG2ID[f'I-{aspect}']
        first = True
        for t_idx in range(len(words)):
            ts, te = positions[t_idx]
            if ts < e and te > s:
                tags[t_idx] = b_id if first else i_id; first = False
    return tags

class ATEDataset(Dataset):
    def __init__(self, items, word2idx, max_len=128):
        self.seqs, self.lens, self.masks, self.tags = [], [], [], []
        for item in items:
            tags = text_to_ate_tags(item['text'], item.get('labels',[]), max_len)
            seq, length = tokenize(item['text'], word2idx, max_len)
            mask = [1]*length + [0]*(max_len-length)
            self.seqs.append(seq); self.lens.append(length)
            self.masks.append(mask); self.tags.append(tags)
        self.seqs = torch.tensor(self.seqs, dtype=torch.long)
        self.lens = torch.tensor(self.lens, dtype=torch.long)
        self.masks = torch.tensor(self.masks, dtype=torch.bool)
        self.tags = torch.tensor(self.tags, dtype=torch.long)
    def __len__(self): return len(self.seqs)
    def __getitem__(self, i):
        return {'seq': self.seqs[i], 'mask': self.masks[i], 'tags': self.tags[i], 'len': self.lens[i]}

# ASC Dataset
class ASCDataset(Dataset):
    def __init__(self, items, word2idx, max_len=128):
        self.seqs, self.lens, self.labels = [], [], []
        d_map = {"POSITIVE": 0, "NEGATIVE": 1, "NEUTRAL": 2}
        for item in items:
            for s, e, raw_label in item.get('labels', []):
                if "#" in raw_label:
                    _, sentiment = raw_label.split("#")
                    if sentiment in d_map:
                        marked = item['text'][:s] + f" [ASP] {item['text'][s:e]} [ASP] " + item['text'][e:]
                        seq, length = tokenize(marked, word2idx, max_len)
                        self.seqs.append(seq); self.lens.append(length)
                        self.labels.append(d_map[sentiment])
        self.seqs = torch.tensor(self.seqs, dtype=torch.long)
        self.lens = torch.tensor(self.lens, dtype=torch.long)
        self.labels = torch.tensor(self.labels, dtype=torch.long)
    def __len__(self): return len(self.seqs)
    def __getitem__(self, i): return {'seq': self.seqs[i], 'len': self.lens[i], 'label': self.labels[i]}

print("Datasets ready!")


## 3. Models (ATE: Seq+CRF, ASC: Classifier)


In [ ]:
# ==================== ATE MODELS ====================
class ATESequenceCRF(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_tags,
                 pretrained_emb=None, n_layers=2, dropout=0.3, pad_idx=0,
                 rnn_type='gru', bidir=True):
        super().__init__()
        self.bidir = bidir
        rnn_out = hidden_dim * 2 if bidir else hidden_dim
        if pretrained_emb is not None:
            self.emb = nn.Embedding.from_pretrained(torch.FloatTensor(pretrained_emb), freeze=False, padding_idx=pad_idx)
        else:
            self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.drop = nn.Dropout(dropout)
        rnn_cls = nn.LSTM if rnn_type=='lstm' else (nn.GRU if rnn_type=='gru' else nn.RNN)
        self.rnn = rnn_cls(emb_dim, hidden_dim, n_layers, batch_first=True,
                           dropout=dropout if n_layers>1 else 0, bidirectional=bidir)
        self.hidden2tag = nn.Sequential(nn.Linear(rnn_out, rnn_out//2), nn.ReLU(),
                                        nn.Dropout(dropout), nn.Linear(rnn_out//2, num_tags))
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, seqs, mask=None, labels=None, lens=None):
        if lens is None: lens = mask.sum(dim=1)
        emb = self.drop(self.emb(seqs))
        packed = nn.utils.rnn.pack_padded_sequence(emb, lens.cpu().clamp(min=1), batch_first=True, enforce_sorted=False)
        output, _ = self.rnn(packed)
        output, _ = nn.utils.rnn.pad_packed_sequence(output, batch_first=True, total_length=seqs.size(1))
        emissions = self.hidden2tag(self.drop(output))
        if labels is not None: return -self.crf(emissions, labels, mask=mask, reduction='mean')
        return self.crf.decode(emissions, mask=mask)

class ATECNNCRF(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_tags,
                 pretrained_emb=None, pad_idx=0, dropout=0.3):
        super().__init__()
        if pretrained_emb is not None:
            self.emb = nn.Embedding.from_pretrained(torch.FloatTensor(pretrained_emb), freeze=False, padding_idx=pad_idx)
        else:
            self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.drop = nn.Dropout(dropout)
        self.convs = nn.ModuleList([nn.Conv1d(emb_dim, hidden_dim, k, padding=k//2) for k in [3,5,7]])
        conv_out = hidden_dim * 3
        self.hidden2tag = nn.Sequential(nn.Linear(conv_out, conv_out//2), nn.ReLU(),
                                        nn.Dropout(dropout), nn.Linear(conv_out//2, num_tags))
        self.crf = CRF(num_tags, batch_first=True)

    def forward(self, seqs, mask=None, labels=None, lens=None):
        emb = self.drop(self.emb(seqs)).transpose(1,2)
        out = torch.cat([torch.relu(c(emb)) for c in self.convs], dim=1).transpose(1,2)
        emissions = self.hidden2tag(self.drop(out))
        if labels is not None: return -self.crf(emissions, labels, mask=mask, reduction='mean')
        return self.crf.decode(emissions, mask=mask)

# ==================== ASC MODELS ====================
class ASCSequenceModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_classes=3,
                 pretrained_emb=None, n_layers=2, dropout=0.3, pad_idx=0,
                 rnn_type='gru', bidir=True):
        super().__init__()
        self.bidir = bidir; self.rnn_type_name = rnn_type
        if pretrained_emb is not None:
            self.emb = nn.Embedding.from_pretrained(torch.FloatTensor(pretrained_emb), freeze=False, padding_idx=pad_idx)
        else:
            self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.drop = nn.Dropout(dropout)
        rnn_cls = nn.LSTM if rnn_type=='lstm' else (nn.GRU if rnn_type=='gru' else nn.RNN)
        self.rnn = rnn_cls(emb_dim, hidden_dim, n_layers, batch_first=True,
                           dropout=dropout if n_layers>1 else 0, bidirectional=bidir)
        out_dim = hidden_dim * 2 if bidir else hidden_dim
        self.fc = nn.Sequential(nn.Linear(out_dim, out_dim//2), nn.ReLU(),
                                nn.Dropout(dropout), nn.Linear(out_dim//2, num_classes))

    def forward(self, x, lens=None):
        emb = self.drop(self.emb(x))
        if lens is not None:
            packed = nn.utils.rnn.pack_padded_sequence(emb, lens.cpu().clamp(min=1), batch_first=True, enforce_sorted=False)
            _, hidden = self.rnn(packed)
        else: _, hidden = self.rnn(emb)
        if self.rnn_type_name == 'lstm': hidden = hidden[0]
        feats = torch.cat((hidden[-2], hidden[-1]), dim=1) if self.bidir else hidden[-1]
        return self.fc(feats)

class ASCCNNModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_classes=3,
                 pretrained_emb=None, pad_idx=0, dropout=0.3):
        super().__init__()
        if pretrained_emb is not None:
            self.emb = nn.Embedding.from_pretrained(torch.FloatTensor(pretrained_emb), freeze=False, padding_idx=pad_idx)
        else:
            self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.drop = nn.Dropout(dropout)
        self.convs = nn.ModuleList([nn.Conv1d(emb_dim, hidden_dim, k, padding=k//2) for k in [3,5,7]])
        self.fc = nn.Sequential(nn.Linear(hidden_dim*3, hidden_dim), nn.ReLU(),
                                nn.Dropout(dropout), nn.Linear(hidden_dim, num_classes))

    def forward(self, x, lens=None):
        emb = self.drop(self.emb(x)).transpose(1,2)
        pooled = [F.max_pool1d(F.relu(c(emb)), emb.size(2)).squeeze(2) for c in self.convs]
        return self.fc(torch.cat(pooled, dim=1))

# Factory functions
def build_ate_model(model_type, **kw):
    if model_type == 'TextCNN':
        return ATECNNCRF(kw['vocab_size'], kw['emb_dim'], kw['hidden_dim'], kw['num_tags'],
                         kw.get('pretrained_emb'), dropout=kw.get('dropout',0.3))
    rnn_map = {'RNN':('rnn',False),'LSTM':('lstm',False),'GRU':('gru',False),'BiLSTM':('lstm',True),'BiGRU':('gru',True)}
    rt, bd = rnn_map[model_type]
    return ATESequenceCRF(kw['vocab_size'], kw['emb_dim'], kw['hidden_dim'], kw['num_tags'],
                          kw.get('pretrained_emb'), kw.get('n_layers',2), kw.get('dropout',0.3), 0, rt, bd)

def build_asc_model(model_type, **kw):
    if model_type == 'TextCNN':
        return ASCCNNModel(kw['vocab_size'], kw['emb_dim'], kw['hidden_dim'], kw.get('num_classes',3),
                           kw.get('pretrained_emb'), dropout=kw.get('dropout',0.3))
    rnn_map = {'RNN':('rnn',False),'LSTM':('lstm',False),'GRU':('gru',False),'BiLSTM':('lstm',True),'BiGRU':('gru',True)}
    rt, bd = rnn_map[model_type]
    return ASCSequenceModel(kw['vocab_size'], kw['emb_dim'], kw['hidden_dim'], kw.get('num_classes',3),
                            kw.get('pretrained_emb'), kw.get('n_layers',2), kw.get('dropout',0.3), 0, rt, bd)

print("All models defined!")


## 4. Training Engine & Metrics


In [ ]:
# ==================== METRICS ====================
def bio_tags_to_spans(tag_ids, bio_tags_list, max_tokens):
    spans = []; cl = None; cs = None
    for t in range(min(len(tag_ids), max_tokens)):
        tn = bio_tags_list[tag_ids[t]] if tag_ids[t] < len(bio_tags_list) else 'O'
        if tn.startswith('B-'):
            if cl: spans.append((cl, cs, t))
            cl = tn[2:]; cs = t
        elif tn.startswith('I-'):
            lb = tn[2:]
            if cl != lb:
                if cl: spans.append((cl, cs, t))
                cl = lb; cs = t
        else:
            if cl: spans.append((cl, cs, t)); cl = None
    if cl: spans.append((cl, cs, len(tag_ids)))
    return spans

def evaluate_spans_f1(pred_spans_list, true_spans_list):
    tp, fp, fn = 0, 0, 0
    for ps, ts in zip(pred_spans_list, true_spans_list):
        ps_set = set(ps); ts_set = set(ts)
        tp += len(ps_set & ts_set); fp += len(ps_set - ts_set); fn += len(ts_set - ps_set)
    p = tp/(tp+fp) if tp+fp>0 else 0; r = tp/(tp+fn) if tp+fn>0 else 0
    f = 2*p*r/(p+r) if p+r>0 else 0
    return {'precision': p, 'recall': r, 'f1': f}

# ==================== TRAINING ENGINE ====================
def train_model_ate(model, train_loader, dev_loader, epochs=30, patience=7, lr=1e-3, name="Model"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    best_score, best_state, wait = 0, None, 0
    history = {'train_loss': [], 'dev_loss': [], 'dev_tok_acc': []}

    print(f"\n{'='*50} Training {name} {'='*50}")
    for ep in range(epochs):
        model.train(); total_loss = 0
        for b in tqdm(train_loader, leave=False):
            optimizer.zero_grad()
            loss = model(b['seq'].to(device), b['mask'].to(device), b['tags'].to(device), b['len'])
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step(); total_loss += loss.item()
        train_loss = total_loss / len(train_loader)

        # Dev eval
        model.eval(); dev_loss = 0; correct = total = 0
        all_pt, all_tt, all_lens = [], [], []
        with torch.no_grad():
            for b in dev_loader:
                l = model(b['seq'].to(device), b['mask'].to(device), b['tags'].to(device), b['len'])
                dev_loss += l.item()
                pt = model(b['seq'].to(device), b['mask'].to(device), lens=b['len'])
                for i in range(len(b['len'])):
                    ln = b['len'][i].item()
                    for p, t in zip(pt[i][:ln], b['tags'][i][:ln].tolist()):
                        if p == t: correct += 1
                        total += 1
                    all_pt.append(pt[i][:ln]); all_tt.append(b['tags'][i][:ln].tolist())
                    all_lens.append(ln)

        dev_loss /= len(dev_loader); tok_acc = correct / max(1, total)
        scheduler.step(dev_loss)
        history['train_loss'].append(train_loss); history['dev_loss'].append(dev_loss)
        history['dev_tok_acc'].append(tok_acc)

        mark = ''
        if tok_acc > best_score:
            best_score = tok_acc; best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}; wait = 0; mark = ' ***'
        else: wait += 1
        print(f"  Ep {ep+1:2d} | TrL: {train_loss:.4f} | DvL: {dev_loss:.4f} | TokAcc: {tok_acc:.4f}{mark}")
        if wait >= patience: print(f"  Early stop"); break
    if best_state: model.load_state_dict(best_state); model.to(device)
    return model, history

def train_model_asc(model, train_loader, dev_loader, epochs=30, patience=7, lr=1e-3, name="Model"):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    best_acc, best_state, wait = 0, None, 0
    history = {'train_loss': [], 'dev_loss': [], 'dev_acc': []}

    print(f"\n{'='*50} Training {name} {'='*50}")
    for ep in range(epochs):
        model.train(); total_loss = 0
        for b in tqdm(train_loader, leave=False):
            optimizer.zero_grad()
            logits = model(b['seq'].to(device))
            loss = criterion(logits, b['label'].to(device))
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step(); total_loss += loss.item()

        model.eval(); dev_loss = 0; preds_all = []; labels_all = []
        with torch.no_grad():
            for b in dev_loader:
                logits = model(b['seq'].to(device))
                loss = criterion(logits, b['label'].to(device)); dev_loss += loss.item()
                preds_all.extend(logits.argmax(1).cpu().tolist())
                labels_all.extend(b['label'].tolist())
        dev_loss /= len(dev_loader)
        dev_acc = np.mean(np.array(preds_all) == np.array(labels_all))
        scheduler.step(dev_loss)
        history['train_loss'].append(total_loss/len(train_loader))
        history['dev_loss'].append(dev_loss); history['dev_acc'].append(dev_acc)

        mark = ''
        if dev_acc > best_acc:
            best_acc = dev_acc; best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}; wait = 0; mark = ' ***'
        else: wait += 1
        print(f"  Ep {ep+1:2d} | TrL: {total_loss/len(train_loader):.4f} | DvL: {dev_loss:.4f} | Acc: {dev_acc:.4f}{mark}")
        if wait >= patience: print(f"  Early stop"); break
    if best_state: model.load_state_dict(best_state); model.to(device)
    return model, history

print("Engine ready!")


## 5. Load & Preprocess Data


In [ ]:
train_items = load_raw_data(os.path.join(DATA_DIR, "train.jsonl"))
dev_items = load_raw_data(os.path.join(DATA_DIR, "dev.jsonl"))
test_items = load_raw_data(os.path.join(DATA_DIR, "test.jsonl"))
for items in [train_items, dev_items, test_items]:
    for item in items: item['text'] = segment_text(item['text'])
print(f"Train: {len(train_items)} | Dev: {len(dev_items)} | Test: {len(test_items)}")

all_texts = [item["text"] for item in train_items + dev_items + test_items]
word2idx = build_vocab(all_texts, min_freq=2)
VOCAB_SIZE = len(word2idx)
EMB_DIM = 150; HIDDEN_DIM = 256; NUM_LAYERS = 2; DROPOUT = 0.3
BATCH_SIZE = 64; EPOCHS = 30; PATIENCE = 7; MAX_LEN = 128
emb_matrix = train_w2v(all_texts, word2idx, emb_dim=EMB_DIM)
print(f"Vocab: {VOCAB_SIZE}")


---
# PART A: Aspect Term Extraction (ATE)
## Train 6 Baselines


In [ ]:
ate_train = DataLoader(ATEDataset(train_items, word2idx, MAX_LEN), BATCH_SIZE, shuffle=True)
ate_dev = DataLoader(ATEDataset(dev_items, word2idx, MAX_LEN), BATCH_SIZE)
ate_test = DataLoader(ATEDataset(test_items, word2idx, MAX_LEN), BATCH_SIZE)

MODEL_TYPES = ["TextCNN", "RNN", "LSTM", "GRU", "BiLSTM", "BiGRU"]
ate_models, ate_histories, ate_results = {}, {}, []

for mt in MODEL_TYPES:
    m = build_ate_model(mt, vocab_size=VOCAB_SIZE, emb_dim=EMB_DIM, hidden_dim=HIDDEN_DIM,
                        num_tags=NUM_ATE_TAGS, pretrained_emb=emb_matrix, n_layers=NUM_LAYERS, dropout=DROPOUT).to(device)
    m, h = train_model_ate(m, ate_train, ate_dev, EPOCHS, PATIENCE, name=f"{mt}-CRF")
    ate_models[mt] = m; ate_histories[mt] = h


## ATE Results


In [ ]:
for mt, m in ate_models.items():
    m.eval(); all_pt, all_tt, all_ln = [], [], []
    with torch.no_grad():
        for b in ate_test:
            pt = m(b['seq'].to(device), b['mask'].to(device), lens=b['len'])
            for i in range(len(b['len'])):
                ln = b['len'][i].item()
                all_pt.append(pt[i][:ln]); all_tt.append(b['tags'][i][:ln].tolist()); all_ln.append(ln)
    ps = [bio_tags_to_spans(p, BIO_TAGS, l) for p,l in zip(all_pt, all_ln)]
    ts = [bio_tags_to_spans(t, BIO_TAGS, l) for t,l in zip(all_tt, all_ln)]
    sf = evaluate_spans_f1(ps, ts)
    ate_results.append({"Model": f"{mt}-CRF", "P": round(sf['precision'],4), "R": round(sf['recall'],4), "F1": round(sf['f1'],4)})

ate_df = pd.DataFrame(ate_results)
display(ate_df.sort_values("F1", ascending=False).style.highlight_max(subset=["F1"], color="lightgreen"))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, h in ate_histories.items():
    axes[0].plot(h['train_loss'], label=name, marker='o', ms=3)
    axes[1].plot(h['dev_tok_acc'], label=name, marker='^', ms=3)
axes[0].set_title('ATE Train Loss'); axes[0].legend(fontsize=8)
axes[1].set_title('ATE Dev Token Acc'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()


---
# PART B: Aspect Sentiment Classification (ASC)
## Train 6 Baselines


In [ ]:
asc_train = DataLoader(ASCDataset(train_items, word2idx, MAX_LEN), BATCH_SIZE, shuffle=True)
asc_dev = DataLoader(ASCDataset(dev_items, word2idx, MAX_LEN), BATCH_SIZE)
asc_test = DataLoader(ASCDataset(test_items, word2idx, MAX_LEN), BATCH_SIZE)
print(f"ASC Samples - Train: {len(asc_train.dataset)} | Dev: {len(asc_dev.dataset)} | Test: {len(asc_test.dataset)}")

asc_models, asc_histories, asc_results = {}, {}, []
for mt in MODEL_TYPES:
    m = build_asc_model(mt, vocab_size=VOCAB_SIZE, emb_dim=EMB_DIM, hidden_dim=HIDDEN_DIM,
                        num_classes=3, pretrained_emb=emb_matrix, n_layers=NUM_LAYERS, dropout=DROPOUT).to(device)
    m, h = train_model_asc(m, asc_train, asc_dev, EPOCHS, PATIENCE, name=mt)
    asc_models[mt] = m; asc_histories[mt] = h


## ASC Results


In [ ]:
CLASS_NAMES = ["POSITIVE", "NEGATIVE", "NEUTRAL"]
criterion = nn.CrossEntropyLoss()

for mt, m in asc_models.items():
    m.eval(); preds, labels = [], []
    with torch.no_grad():
        for b in asc_test:
            logits = m(b['seq'].to(device))
            preds.extend(logits.argmax(1).cpu().tolist())
            labels.extend(b['label'].tolist())
    acc = np.mean(np.array(preds) == np.array(labels))
    mf1 = f1_score(labels, preds, average='macro')
    asc_results.append({"Model": mt, "Accuracy": round(acc,4), "Macro_F1": round(mf1,4)})

asc_df = pd.DataFrame(asc_results)
display(asc_df.sort_values("Macro_F1", ascending=False).style.highlight_max(subset=["Macro_F1"], color="lightgreen"))

# Confusion matrix best model
best_asc = asc_df.loc[asc_df["Macro_F1"].idxmax(), "Model"]
m = asc_models[best_asc]; m.eval(); preds, labels = [], []
with torch.no_grad():
    for b in asc_test:
        preds.extend(m(b['seq'].to(device)).argmax(1).cpu().tolist())
        labels.extend(b['label'].tolist())
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title(f'Confusion Matrix: {best_asc}'); ax.set_ylabel('True'); ax.set_xlabel('Pred')
plt.tight_layout(); plt.show()


## Save All Results


In [ ]:
os.makedirs(os.path.join(SAVE_DIR, "ate"), exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, "asc"), exist_ok=True)

ate_df.to_csv(os.path.join(SAVE_DIR, "ate", "ate_results.csv"), index=False)
asc_df.to_csv(os.path.join(SAVE_DIR, "asc", "asc_results.csv"), index=False)

best_ate_name = ate_df.loc[ate_df["F1"].idxmax(), "Model"].replace("-CRF","")
torch.save(ate_models[best_ate_name].state_dict(), os.path.join(SAVE_DIR, "ate", f"best_ate.pt"))
torch.save(asc_models[best_asc].state_dict(), os.path.join(SAVE_DIR, "asc", f"best_asc.pt"))

print(f"\n{'='*60}")
print(f"RESULTS SUMMARY")
print(f"{'='*60}")
print(f"Best ATE: {best_ate_name}-CRF (F1={ate_df['F1'].max():.4f})")
print(f"Best ASC: {best_asc} (Macro F1={asc_df['Macro_F1'].max():.4f})")
print(f"\nAll saved to {SAVE_DIR}/")
print("Done! Download results from Output tab.")
